## Import libraries

In [ ]:
import sys, os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Connect to updated utils.py
sys.path.insert(0, os.path.abspath('..'))
from utils import load_metabric_data, SmartClinicalImputer, DiscordantSignatureAdder

df = load_metabric_data()
df_LumA = df[df['pam50_+_claudin-low_subtype'] == 'LumA'].copy()

In [ ]:
# Define Target
df_LumA['target_mortality'] = df_LumA['death_from_cancer'].apply(
    lambda x: 1 if str(x).strip().lower() == 'died of disease' else 0
)

# Drop Leakage
leakage_cols = ['patient_id', 'overall_survival_months', 'overall_survival', 
                'death_from_cancer', 'target_mortality', 'chemotherapy', 
                'hormone_therapy', 'radio_therapy', 'type_of_breast_surgery']
X_raw = df_LumA.drop(columns=[col for col in leakage_cols if col in df_LumA.columns])
y = df_LumA['target_mortality']

# Encoding clinical & mutations
for col in ["er_status", "her2_status", "pr_status"]:
    if col in X_raw.columns: X_raw[col] = (X_raw[col] == "Positive").astype(int)
if "cellularity" in X_raw.columns: 
    X_raw["cellularity"] = X_raw["cellularity"].map({"Low": 0, "Moderate": 1, "High": 2})

mutation_cols = [c for c in X_raw.columns if c.endswith("_mut")]
for col in mutation_cols:
    X_raw[col] = X_raw[col].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == "0" else 1)

X = X_raw.select_dtypes(include=['number'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing Pipeline 
pipeline = Pipeline(steps=[
    ("smart_imputer", SmartClinicalImputer()),          
    ("median_imputer", SimpleImputer(strategy="median")),      
    ("scaler", StandardScaler()),                       
    ("molecular_score", DiscordantSignatureAdder())     
])

preprocessor = ColumnTransformer(transformers=[("num", pipeline, X.columns.tolist())])
preprocessor.set_output(transform="pandas")

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"✅ Preprocessing completed. Final features: {X_train_transformed.shape[1]}")
X_train_transformed.head()